# LeetCode 217 — Contains Duplicate bằng Python

**Link đề:** https://leetcode.com/problems/contains-duplicate/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Sorting

```python
class Solution:
    def containsDuplicate(self, nums: list[int]) -> bool:
        ...
```

## Mục tiêu học
1. Đi từ cách thử mọi cặp `O(n²)` → sắp xếp `O(n log n)` → **hash set** `O(n)`.
2. Hiểu `set`: thêm, kiểm tra tồn tại, vì sao tra cứu trung bình `O(1)`.
3. Hiểu `sorted` / `.sort()`, so sánh phần tử kề nhau, `len(set(nums))`, `collections.Counter`.
4. Trace từng bước "kiểm tra trước, lưu sau"; chứng minh tính đúng; đo thời gian thực tế.
5. Chuẩn bị cho các biến thể: Contains Duplicate II (khoảng cách ≤ k), tìm phần tử trùng đầu tiên.

## 1. Đề bài

Cho mảng số nguyên `nums`. Trả `True` nếu **có giá trị xuất hiện ít nhất hai lần**, trả `False` nếu mọi phần tử đều khác nhau.

| `nums` | Kết quả | Vì sao |
|---|:---:|---|
| `[1, 2, 3, 1]` | `True` | `1` ở chỉ số 0 và 3 |
| `[1, 2, 3, 4]` | `False` | tất cả khác nhau |
| `[1, 1, 1, 3, 3, 4, 3, 2, 4, 2]` | `True` | nhiều số lặp |

**Constraints:** `1 <= len(nums) <= 10^5`; `-10^9 <= nums[i] <= 10^9`.

Câu hỏi then chốt: khi đứng ở phần tử `x`, làm sao **nhanh chóng** biết `x` đã xuất hiện trước đó chưa?

## 2. Nền tảng Python

### 2.1. List, chỉ số, `range`, `len`

`nums[i]` đọc phần tử thứ `i` (bắt đầu từ 0). `range(a, b)` sinh `a, a+1, ..., b-1`. `len(nums)` là số phần tử.

In [1]:
nums = [1, 2, 3, 1]
print("len =", len(nums), "| nums[0] =", nums[0], "| nums[3] =", nums[3], "| nums[-1] =", nums[-1])
print("range(1, 4) ->", list(range(1, 4)))

len = 4 | nums[0] = 1 | nums[3] = 1 | nums[-1] = 1
range(1, 4) -> [1, 2, 3]


### 2.2. Vòng lặp lồng nhau để duyệt mọi cặp `i < j`

`for i in range(n)` ngoài, `for j in range(i + 1, n)` trong → mỗi cặp chỉ xét **một lần** và không bao giờ `i == j` (một phần tử không được tính là trùng với chính nó).

In [2]:
nums = [5, 7, 9]
for i in range(len(nums)):
    for j in range(i + 1, len(nums)):
        print(f"cặp (i={i}, j={j}) -> ({nums[i]}, {nums[j]})")

cặp (i=0, j=1) -> (5, 7)
cặp (i=0, j=2) -> (5, 9)
cặp (i=1, j=2) -> (7, 9)


## 3. Cách 1 — Brute Force: thử mọi cặp

Ý nghĩ tự nhiên: "so mỗi phần tử với mọi phần tử đứng sau nó; thấy bằng nhau thì có trùng".

Số cặp là `n(n-1)/2`. Với `n = 10^5`: khoảng **5 tỷ** phép so sánh → quá chậm (Time Limit Exceeded).

In [3]:
def contains_duplicate_brute(nums: list[int]) -> bool:
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] == nums[j]:
                return True
    return False

print(contains_duplicate_brute([1, 2, 3, 1]), contains_duplicate_brute([1, 2, 3, 4]))
n = 10**5
print(f"Số cặp khi n = {n}: {n * (n - 1) // 2:,}")

True False
Số cặp khi n = 100000: 4,999,950,000


## 4. Cách 2 — Sắp xếp rồi so phần tử kề nhau

**Insight 1:** sau khi sắp xếp, các giá trị bằng nhau **đứng cạnh nhau**. Chỉ cần so `nums[i]` với `nums[i + 1]`.

```text
[1, 2, 3, 1] --sắp xếp--> [1, 1, 2, 3]   -> nums[0] == nums[1] -> True
```

### 4.1. `sorted(nums)` và `nums.sort()`

- `sorted(nums)` trả về **list mới** đã sắp xếp, list gốc không đổi.
- `nums.sort()` sắp xếp **tại chỗ** (thay đổi list gốc), trả `None`.

Thời gian sắp xếp: `O(n log n)` (thuật toán Timsort của Python).

In [4]:
nums = [3, 1, 2, 1]
s = sorted(nums)
print("sorted:", s, "| gốc vẫn là:", nums)
r = nums.sort()
print("sau .sort():", nums, "| giá trị trả về của .sort():", r)

sorted: [1, 1, 2, 3] | gốc vẫn là: [3, 1, 2, 1]
sau .sort(): [1, 1, 2, 3] | giá trị trả về của .sort(): None


In [5]:
def contains_duplicate_sort(nums: list[int]) -> bool:
    s = sorted(nums)                     # không làm hỏng input của người gọi
    for i in range(len(s) - 1):          # len - 1 vì đọc s[i + 1]
        if s[i] == s[i + 1]:
            return True
    return False

for nums in [[1, 2, 3, 1], [1, 2, 3, 4], [1, 1, 1, 3, 3, 4, 3, 2, 4, 2], [7]]:
    print(nums, "->", contains_duplicate_sort(nums))

[1, 2, 3, 1] -> True
[1, 2, 3, 4] -> False
[1, 1, 1, 3, 3, 4, 3, 2, 4, 2] -> True
[7] -> False


**Tại sao `range(len(s) - 1)`?** Ta đọc `s[i + 1]`; nếu `i` chạy tới chỉ số cuối thì `s[i + 1]` vượt biên → `IndexError`. Với mảng 1 phần tử, `range(0)` rỗng → trả `False` đúng.

**Hạn chế:** vẫn `O(n log n)`, và phải sắp xếp toàn bộ ngay cả khi cặp trùng nằm ngay đầu mảng.

## 5. Insight 2 — chỉ cần nhớ "đã thấy những gì"

Duyệt từ trái sang phải. Tại `x`, câu hỏi chỉ là: **"`x` đã xuất hiện ở bên trái chưa?"**

Nếu có một "cuốn sổ" ghi các số đã thấy và tra cứu được trong `O(1)`, ta giải bài trong **một lượt** `O(n)`. Cuốn sổ đó là **hash set**.

## 6. Nền tảng: `set` — tập hợp băm (hash set)

| Thao tác | Cú pháp | Thời gian (trung bình) |
|---|---|---|
| Tạo rỗng | `seen = set()` | |
| Thêm | `seen.add(x)` | `O(1)` |
| Kiểm tra | `x in seen` | `O(1)` |
| Kích thước | `len(seen)` | `O(1)` |
| Tạo từ list | `set(nums)` | `O(n)` |

**Vì sao nhanh?** Python tính `hash(x)` — một số nguyên — để biết `x` nằm ở "ngăn" nào trong bảng. Tra cứu chỉ cần nhìn vào ngăn đó thay vì quét toàn bộ. Trường hợp xấu (nhiều phần tử đụng ngăn — collision) có thể chậm hơn, nên ta nói `O(1)` **trung bình**.

So sánh: `x in some_list` phải quét từng phần tử → `O(n)`.

In [6]:
seen = set()
for x in [1, 2, 3]:
    seen.add(x)
print(seen, "| 2 in seen:", 2 in seen, "| 9 in seen:", 9 in seen)
seen.add(2)
print("Thêm 2 lần nữa không đổi:", seen, "len =", len(seen))
print("set([1,2,3,1]) =", set([1, 2, 3, 1]))
print("hash(42) =", hash(42), "| hash('abc') là một số nguyên:", isinstance(hash("abc"), int))

{1, 2, 3} | 2 in seen: True | 9 in seen: False
Thêm 2 lần nữa không đổi: {1, 2, 3} len = 3
set([1,2,3,1]) = {1, 2, 3}
hash(42) = 42 | hash('abc') là một số nguyên: True


In [7]:
import time
data_list = list(range(200_000))
data_set = set(data_list)
target = -1                                # không có -> phải quét hết list

t0 = time.perf_counter()
for _ in range(200):
    target in data_list
t1 = time.perf_counter()
for _ in range(200):
    target in data_set
t2 = time.perf_counter()
print(f"200 lần 'in list': {t1 - t0:.4f}s | 200 lần 'in set': {t2 - t1:.6f}s")
print("set nhanh hơn nhiều lần:", (t1 - t0) > (t2 - t1))

200 lần 'in list': 0.2223s | 200 lần 'in set': 0.000084s
set nhanh hơn nhiều lần: True


## 7. Cách 3 — Lời giải chính: hash set, dừng sớm

Từng đoạn:

**Đoạn 1:** `seen = set()` — chưa thấy gì.

**Đoạn 2:** duyệt `for x in nums:`.

**Đoạn 3 — kiểm tra TRƯỚC:**
```python
    if x in seen:
        return True     # x đã xuất hiện bên trái -> trùng
```

**Đoạn 4 — lưu SAU:**
```python
    seen.add(x)
```

**Đoạn 5:** duyệt hết không trùng → `return False`.

Vì sao "kiểm tra trước, lưu sau"? Nếu lưu `x` trước rồi mới kiểm tra, `x in seen` **luôn** đúng — chính `x` vừa thêm bị coi là trùng với nó.

In [8]:
def contains_duplicate_set(nums: list[int]) -> bool:
    seen = set()
    for x in nums:
        if x in seen:
            return True
        seen.add(x)
    return False

for nums in [[1, 2, 3, 1], [1, 2, 3, 4], [1, 1, 1, 3, 3, 4, 3, 2, 4, 2], [7]]:
    print(nums, "->", contains_duplicate_set(nums))

[1, 2, 3, 1] -> True
[1, 2, 3, 4] -> False
[1, 1, 1, 3, 3, 4, 3, 2, 4, 2] -> True
[7] -> False


## 8. TRACE

### 8.1. Bảng trace `nums = [1, 2, 3, 1]`

| i | `x` | `seen` trước | `x in seen`? | Hành động |
|---:|:-:|---|:-:|---|
| 0 | 1 | `{}` | không | thêm 1 |
| 1 | 2 | `{1}` | không | thêm 2 |
| 2 | 3 | `{1, 2}` | không | thêm 3 |
| 3 | 1 | `{1, 2, 3}` | **có** | trả `True` ngay |

In [9]:
def trace_contains_duplicate(nums: list[int]) -> bool:
    seen = set()
    for i, x in enumerate(nums):
        before = sorted(seen)
        if x in seen:
            print(f"i={i}: x={x:>2}, seen trước={before} -> ĐÃ THẤY -> True")
            return True
        seen.add(x)
        print(f"i={i}: x={x:>2}, seen trước={before} -> chưa thấy, thêm {x}")
    print("Duyệt hết, không trùng -> False")
    return False

trace_contains_duplicate([1, 2, 3, 1])
print()
trace_contains_duplicate([1, 2, 3, 4])

i=0: x= 1, seen trước=[] -> chưa thấy, thêm 1
i=1: x= 2, seen trước=[1] -> chưa thấy, thêm 2
i=2: x= 3, seen trước=[1, 2] -> chưa thấy, thêm 3
i=3: x= 1, seen trước=[1, 2, 3] -> ĐÃ THẤY -> True

i=0: x= 1, seen trước=[] -> chưa thấy, thêm 1
i=1: x= 2, seen trước=[1] -> chưa thấy, thêm 2
i=2: x= 3, seen trước=[1, 2] -> chưa thấy, thêm 3
i=3: x= 4, seen trước=[1, 2, 3] -> chưa thấy, thêm 4
Duyệt hết, không trùng -> False


False

### 8.2. Trace ví dụ 3 — dừng rất sớm

`[1, 1, 1, 3, ...]` dừng ngay ở `i = 1`, không cần đọc 8 phần tử còn lại. Đây là lợi thế của cách 3 so với cách sắp xếp (luôn sắp toàn bộ) và cách `len(set)` (luôn tạo set toàn bộ).

In [10]:
trace_contains_duplicate([1, 1, 1, 3, 3, 4, 3, 2, 4, 2])

i=0: x= 1, seen trước=[] -> chưa thấy, thêm 1
i=1: x= 1, seen trước=[1] -> ĐÃ THẤY -> True


True

## 9. Cách 4 — One-liner `len(set(nums)) != len(nums)`

`set(nums)` bỏ hết phần tử trùng. Nếu số phần tử giảm → có trùng.

```text
nums      = [1, 2, 3, 1]   len = 4
set(nums) = {1, 2, 3}      len = 3   -> 3 != 4 -> True
```

Ngắn, rõ, `O(n)` time, `O(n)` space. Nhược điểm: luôn xử lý **toàn bộ** mảng, không dừng sớm.

In [11]:
def contains_duplicate_len(nums: list[int]) -> bool:
    return len(set(nums)) != len(nums)

for nums in [[1, 2, 3, 1], [1, 2, 3, 4]]:
    print(nums, "| set =", set(nums), "->", contains_duplicate_len(nums))

[1, 2, 3, 1] | set = {1, 2, 3} -> True
[1, 2, 3, 4] | set = {1, 2, 3, 4} -> False


## 10. Cách 5 — Đếm tần suất bằng `collections.Counter`

`Counter(nums)` là một dict đặc biệt: khóa là giá trị, value là **số lần xuất hiện**. Có trùng ⇔ có tần suất `>= 2`.

`any(điều_kiện for ...)` trả `True` nếu **ít nhất một** phần tử thỏa (và dừng ngay khi gặp).

Cách này hữu ích khi đề hỏi thêm "số nào trùng", "trùng bao nhiêu lần".

In [12]:
from collections import Counter

cnt = Counter([1, 1, 1, 3, 3, 4, 3, 2, 4, 2])
print(cnt)
print("cnt[3] =", cnt[3], "| cnt[99] =", cnt[99], "(khóa không có -> 0, không lỗi)")
print("any([False, True, False]) =", any([False, True, False]))

def contains_duplicate_counter(nums: list[int]) -> bool:
    return any(c >= 2 for c in Counter(nums).values())

print(contains_duplicate_counter([1, 2, 3, 1]), contains_duplicate_counter([1, 2, 3, 4]))

Counter({1: 3, 3: 3, 4: 2, 2: 2})
cnt[3] = 3 | cnt[99] = 0 (khóa không có -> 0, không lỗi)
any([False, True, False]) = True
True False


## 11. Code nộp LeetCode

In [13]:
class Solution:
    def containsDuplicate(self, nums: list[int]) -> bool:
        seen = set()
        for x in nums:
            if x in seen:
                return True
            seen.add(x)
        return False

sol = Solution()
print(sol.containsDuplicate([1, 2, 3, 1]), sol.containsDuplicate([1, 2, 3, 4]),
      sol.containsDuplicate([1, 1, 1, 3, 3, 4, 3, 2, 4, 2]))

True False True


Bản ngắn cũng được chấp nhận:

```python
class Solution:
    def containsDuplicate(self, nums: list[int]) -> bool:
        return len(set(nums)) != len(nums)
```

## 12. Vì sao đúng? (invariant)

**Cách 3** — ở đầu vòng thứ `i`:

> `seen` chứa đúng tập các giá trị `nums[0..i-1]`, và các giá trị này **đôi một khác nhau**.

- **Ban đầu** (`i = 0`): `seen` rỗng → đúng.
- **Nếu `nums[i] in seen`:** tồn tại `j < i` với `nums[j] == nums[i]` → có trùng → trả `True` đúng.
- **Nếu không:** `nums[i]` khác mọi giá trị trước → thêm vào `seen`, invariant đúng với `i + 1`.
- **Kết thúc vòng lặp:** invariant với `i = n` nói mọi phần tử đôi một khác nhau → `False` đúng.

**Cách 2** — nếu có `nums[a] == nums[b]` thì sau sắp xếp, mọi phần tử nằm giữa chúng cũng bằng giá trị đó (vì dãy không giảm), nên chắc chắn có **hai phần tử kề nhau** bằng nhau. Ngược lại, hai phần tử kề bằng nhau hiển nhiên là trùng.

## 13. Time / Space — so sánh

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Time | Space phụ | Dừng sớm? |
|---|---|---|---|:---:|
| 1. Brute force | mọi cặp `i < j` | `O(n²)` | `O(1)` | có |
| 2. Sắp xếp | so phần tử kề | `O(n log n)` | `O(n)` với `sorted` (hoặc `O(1)`* nếu `.sort()` tại chỗ) | sau khi sắp |
| 3. Hash set | kiểm tra trước, lưu sau | `O(n)` trung bình | `O(n)` | **có** |
| 4. `len(set)` | so độ dài | `O(n)` trung bình | `O(n)` | không |
| 5. `Counter` | đếm tần suất | `O(n)` trung bình | `O(n)` | không |

\* Timsort cần bộ nhớ tạm tới `O(n)` trong trường hợp xấu; trong phỏng vấn thường nói "sắp xếp tại chỗ, bộ nhớ phụ nhỏ".

**Đánh đổi:** cách 2 tiết kiệm bộ nhớ hơn nếu được phép sửa input; cách 3 nhanh nhất về Big-O.

In [14]:
import time, random
rng = random.Random(217)
big = rng.sample(range(-10**9, 10**9), 10**5)       # 10^5 số khác nhau -> worst case (không trùng)
for name, f in [("sort", contains_duplicate_sort), ("set", contains_duplicate_set),
                ("len(set)", contains_duplicate_len), ("Counter", contains_duplicate_counter)]:
    t0 = time.perf_counter(); r = f(big); t1 = time.perf_counter()
    print(f"{name:<9} -> {r} trong {t1 - t0:.4f}s")
small = big[:3000]
t0 = time.perf_counter(); contains_duplicate_brute(small); t1 = time.perf_counter()
print(f"brute với chỉ 3000 phần tử: {t1 - t0:.4f}s (với 10^5 sẽ chậm hơn ~1100 lần)")

sort      -> False trong 0.0241s
set       -> False trong 0.0091s
len(set)  -> False trong 0.0051s
Counter   -> False trong 0.0153s


brute với chỉ 3000 phần tử: 0.1339s (với 10^5 sẽ chậm hơn ~1100 lần)


## 14. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("ví dụ 1", sol.containsDuplicate([1, 2, 3, 1]), True)
check("ví dụ 2", sol.containsDuplicate([1, 2, 3, 4]), False)
check("ví dụ 3", sol.containsDuplicate([1, 1, 1, 3, 3, 4, 3, 2, 4, 2]), True)
check("1 phần tử", sol.containsDuplicate([5]), False)
check("2 phần tử trùng", sol.containsDuplicate([5, 5]), True)
check("số âm trùng", sol.containsDuplicate([-1, 0, -1]), True)
check("0 và -0 (cùng giá trị)", sol.containsDuplicate([0, -0]), True)
check("biên giá trị", sol.containsDuplicate([-10**9, 10**9]), False)
check("trùng ở cuối", sol.containsDuplicate(list(range(10**5 - 1)) + [0]), True)
check("10^5 phần tử khác nhau", sol.containsDuplicate(list(range(10**5))), False)

PASS ví dụ 1: True
PASS ví dụ 2: False
PASS ví dụ 3: True
PASS 1 phần tử: False
PASS 2 phần tử trùng: True
PASS số âm trùng: True
PASS 0 và -0 (cùng giá trị): True
PASS biên giá trị: False
PASS trùng ở cuối: True
PASS 10^5 phần tử khác nhau: False


In [16]:
rng = random.Random(2017)
fns = [contains_duplicate_brute, contains_duplicate_sort, contains_duplicate_set,
       contains_duplicate_len, contains_duplicate_counter, Solution().containsDuplicate]
for _ in range(3000):
    nums = [rng.randint(-6, 6) for _ in range(rng.randint(1, 10))]
    results = {f(nums) for f in fns}
    assert len(results) == 1, (nums, results)
print("6 cách khớp nhau trên 3000 mảng ngẫu nhiên")

6 cách khớp nhau trên 3000 mảng ngẫu nhiên


## 15. Lỗi người mới hay gặp

### Lỗi 1 — Lưu trước, kiểm tra sau

In [17]:
def contains_duplicate_bug_order(nums):
    seen = set()
    for x in nums:
        seen.add(x)          # SAI: thêm trước
        if x in seen:        # luôn True
            return True
    return False

print("Bản lỗi [1,2,3,4]:", contains_duplicate_bug_order([1, 2, 3, 4]), "(đúng phải là False)")

Bản lỗi [1,2,3,4]: True (đúng phải là False)


### Lỗi 2 — Dùng `list` làm "sổ đã thấy"

```python
seen = []
if x in seen: ...      # O(n) mỗi lần -> tổng O(n²)
seen.append(x)
```
Kết quả đúng nhưng chậm như brute force. Dùng `set()`.

### Lỗi 3 — Brute force cho `j` bắt đầu từ 0

```python
for i in range(n):
    for j in range(n):          # SAI: có cặp i == j
        if nums[i] == nums[j]:
            return True
```
Luôn trả `True` vì `nums[i] == nums[i]`. Sửa: `range(i + 1, n)`.

In [18]:
def contains_duplicate_bug_same_index(nums):
    n = len(nums)
    for i in range(n):
        for j in range(n):
            if nums[i] == nums[j]:
                return True
    return False

print("Bản lỗi i==j, [1,2,3]:", contains_duplicate_bug_same_index([1, 2, 3]), "(đúng phải là False)")

Bản lỗi i==j, [1,2,3]: True (đúng phải là False)


### Lỗi 4 — Vượt biên khi so phần tử kề

`for i in range(len(s)): if s[i] == s[i + 1]` → `IndexError` ở phần tử cuối. Dùng `range(len(s) - 1)`.

### Lỗi 5 — `nums.sort()` rồi gán kết quả

`s = nums.sort()` khiến `s = None` (vì `.sort()` trả `None`). Muốn list mới thì dùng `sorted(nums)`.

In [19]:
s = [3, 1, 2]
try:
    for i in range(len(s)):
        if s[i] == s[i + 1]:
            pass
except IndexError as e:
    print("IndexError:", e)

nums = [3, 1, 2]
s = nums.sort()
print("s = nums.sort() ->", s)

IndexError: list index out of range
s = nums.sort() -> None


## 16. Mẹo nhận dạng pattern

- "Có phần tử nào **xuất hiện lại**", "đã thấy chưa", "distinct / unique" → **hash set**.
- Cần thêm **vị trí** lần thấy trước → **hash map** `giá_trị → chỉ số` (Two Sum, Contains Duplicate II).
- Cần **số lần xuất hiện** → `Counter`.
- Không được dùng thêm bộ nhớ → **sắp xếp** rồi xét kề nhau.
- Khuôn "kiểm tra trước, lưu sau" giống hệt Two Sum (1) — chỉ khác câu hỏi tra cứu là `x` thay vì `target - x`.

## 17. Biến thể

### 17.1. Trả về giá trị trùng **đầu tiên** (theo vị trí lần lặp thứ hai)
Cùng khuôn set, trả `x` thay vì `True`.

### 17.2. Contains Duplicate II (LeetCode 219) — trùng với khoảng cách chỉ số `<= k`
Thay set bằng dict `giá_trị → chỉ số gần nhất`.

### 17.3. Liệt kê mọi giá trị xuất hiện ≥ 2 lần
Dùng `Counter`.

In [20]:
def first_repeated(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return x
        seen.add(x)
    return None

def contains_nearby_duplicate(nums, k):
    last = {}
    for i, x in enumerate(nums):
        if x in last and i - last[x] <= k:
            return True
        last[x] = i
    return False

def all_duplicates(nums):
    return sorted(v for v, c in Counter(nums).items() if c >= 2)

print("first_repeated([3,1,4,1,5,3]) =", first_repeated([3, 1, 4, 1, 5, 3]))
print("219: [1,2,3,1], k=3 ->", contains_nearby_duplicate([1, 2, 3, 1], 3),
      "| [1,2,3,1,2,3], k=2 ->", contains_nearby_duplicate([1, 2, 3, 1, 2, 3], 2))
print("all_duplicates =", all_duplicates([1, 1, 1, 3, 3, 4, 3, 2, 4, 2]))

first_repeated([3,1,4,1,5,3]) = 1
219: [1,2,3,1], k=3 -> True | [1,2,3,1,2,3], k=2 -> False
all_duplicates = [1, 2, 3, 4]


## 18. Bài tập tự luyện

### Bài 1 — Dự đoán output
`len(set([2, 2, 2]))`, `sorted([3, -1, 3])`, `Counter("abca")['a']`, `5 in {1, 5}`.

### Bài 2 — Tính tay
Lập bảng trace cách set cho `[4, 7, 2, 7, 4]`. Dừng ở chỉ số nào?

### Bài 3 — TODO: viết cách hash set

In [21]:
def exercise_contains_duplicate(nums: list[int]) -> bool:
    # TODO 1: tạo seen
    # TODO 2: duyệt x: nếu đã thấy -> True; chưa thì thêm
    # TODO 3: trả False
    pass

# assert exercise_contains_duplicate([1, 2, 3, 1]) is True
# assert exercise_contains_duplicate([1, 2, 3, 4]) is False

### Bài 4 — TODO: viết cách sắp xếp (không dùng set/dict)

In [22]:
def exercise_contains_duplicate_sort(nums: list[int]) -> bool:
    # TODO: sorted + so phần tử kề, chú ý range(len - 1)
    pass

# assert exercise_contains_duplicate_sort([1, 1, 1, 3]) is True
# assert exercise_contains_duplicate_sort([9]) is False

### Bài 5 — Biến thể: trả về **chỉ số** của hai phần tử trùng đầu tiên
Trả `(j, i)` với `j < i`, `nums[j] == nums[i]`, `i` nhỏ nhất có thể; không có thì trả `None`.
`first_duplicate_pair([3, 1, 4, 1, 5, 3])` → `(1, 3)`.

In [23]:
def exercise_first_duplicate_pair(nums: list[int]):
    # TODO: dùng dict giá_trị -> chỉ số
    pass

# assert exercise_first_duplicate_pair([3, 1, 4, 1, 5, 3]) == (1, 3)
# assert exercise_first_duplicate_pair([1, 2]) is None

### Bài 6 — Mini challenge: chuỗi có ký tự lặp không? (không phân biệt hoa thường, bỏ qua khoảng trắng)
`has_repeated_letter("Ab c")` → `False`; `has_repeated_letter("Hello")` → `True`; `has_repeated_letter("a A")` → `True`.

In [24]:
def exercise_has_repeated_letter(s: str) -> bool:
    # TODO: chuẩn hóa ký tự (lower), bỏ ' ', rồi áp dụng khuôn set
    pass

# assert exercise_has_repeated_letter("Ab c") is False
# assert exercise_has_repeated_letter("Hello") is True
# assert exercise_has_repeated_letter("a A") is True

## Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | set bỏ trùng; sorted tăng dần | Counter đếm | `1`, `[-1, 3, 3]`, `2`, `True` |
| 2 | Kiểm tra trước, lưu sau | ghi `seen` mỗi bước | dừng ở `i = 3` (x = 7) |
| 3 | Cuốn sổ đã thấy | `seen = set()` | `if x in seen: return True` rồi `seen.add(x)` |
| 4 | Trùng thì đứng cạnh nhau | `s = sorted(nums)` | `for i in range(len(s) - 1): if s[i] == s[i+1]` |
| 5 | Cần nhớ vị trí | `pos = {}` | `if x in pos: return (pos[x], i)` |
| 6 | Biến đổi trước khi kiểm tra | `ch.lower()`, `if ch == ' ': continue` | dùng lại khuôn set |

## Đáp án tham khảo

**Bài 1:** `1`, `[-1, 3, 3]`, `2`, `True`.

**Bài 2:**

| i | x | seen trước | hành động |
|---:|:-:|---|---|
| 0 | 4 | {} | thêm 4 |
| 1 | 7 | {4} | thêm 7 |
| 2 | 2 | {4, 7} | thêm 2 |
| 3 | 7 | {2, 4, 7} | **đã thấy → True**, dừng ở `i = 3` |

In [25]:
def answer_contains_duplicate(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return True
        seen.add(x)
    return False

def answer_contains_duplicate_sort(nums):
    s = sorted(nums)
    for i in range(len(s) - 1):
        if s[i] == s[i + 1]:
            return True
    return False

def answer_first_duplicate_pair(nums):
    pos = {}
    for i, x in enumerate(nums):
        if x in pos:
            return (pos[x], i)
        pos[x] = i
    return None

def answer_has_repeated_letter(s):
    seen = set()
    for ch in s:
        if ch == ' ':
            continue                  # bỏ qua, sang ký tự tiếp
        ch = ch.lower()
        if ch in seen:
            return True
        seen.add(ch)
    return False

assert len(set([2, 2, 2])) == 1 and sorted([3, -1, 3]) == [-1, 3, 3]
assert Counter("abca")['a'] == 2 and (5 in {1, 5}) is True
assert trace_contains_duplicate([4, 7, 2, 7, 4]) is True
assert answer_contains_duplicate([1, 2, 3, 1]) is True and answer_contains_duplicate([1, 2, 3, 4]) is False
assert answer_contains_duplicate_sort([1, 1, 1, 3]) is True and answer_contains_duplicate_sort([9]) is False
assert answer_first_duplicate_pair([3, 1, 4, 1, 5, 3]) == (1, 3) and answer_first_duplicate_pair([1, 2]) is None
assert answer_has_repeated_letter("Ab c") is False
assert answer_has_repeated_letter("Hello") is True and answer_has_repeated_letter("a A") is True
print("Đáp án tham khảo PASS toàn bộ!")

i=0: x= 4, seen trước=[] -> chưa thấy, thêm 4
i=1: x= 7, seen trước=[4] -> chưa thấy, thêm 7
i=2: x= 2, seen trước=[4, 7] -> chưa thấy, thêm 2
i=3: x= 7, seen trước=[2, 4, 7] -> ĐÃ THẤY -> True
Đáp án tham khảo PASS toàn bộ!


## Template nhớ nhanh + checklist submit

```python
seen = set()
for x in nums:
    if x in seen:      # 1. KIỂM TRA trước
        return True
    seen.add(x)        # 2. LƯU sau
return False
```

Ngắn gọn: `return len(set(nums)) != len(nums)`.

**Checklist:**
- [ ] Dùng `set`, không dùng `list` làm sổ đã thấy.
- [ ] Kiểm tra trước khi thêm.
- [ ] Nếu sắp xếp: `range(len(s) - 1)`, dùng `sorted` nếu không muốn sửa input.
- [ ] Đã thử mảng 1 phần tử, trùng ở đầu / cuối, số âm.
- [ ] Tên method `containsDuplicate`, trả `bool`.